# b3 · Elastic Net forecasts

One-month-ahead log realized variance for the five cyclical ETFs. Eight Elastic Net variants share one walk-forward engine: expanding window, monthly refit, 60-month burn-in.

| Variant | Features |
|---|---|
| `en_full_macro` | 11 macro variables + `log_rv_1m` |
| `en_no_vix` | full macro without `VIXCLS` |
| `en_vix_only` | `VIXCLS` + `log_rv_1m` |
| `en_no_macro` | `log_rv_1m` only |
| `*_har` | same four, with `log_rv_1m`, `log_rv_avg_3m`, `log_rv_avg_12m` |

Output: `forecasts/en_forecasts.csv`. Tests are in `evaluation/e3_vix_effect_en.ipynb`.

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

PROJECT_ROOT = "../../.."   # models/forecasting/build -> repo root
MACRO_PATH = os.path.join(PROJECT_ROOT, "dataset", "macro_variables", "final_clean", "final_macro_data.csv")
RV_PATH = os.path.join(PROJECT_ROOT, "dataset", "price", "clean", "realized_variance.csv")
FORECASTS_DIR = os.path.join(PROJECT_ROOT, "models", "forecasting", "forecasts")

CYCLICAL_TICKERS = ["XLI", "XLB", "XLE", "XLY", "XLF"]
BURN_IN = 60        # months of history before the first forecast
REFIT_EVERY = 1     # refit and re-tune every month
L1_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]

## Data

The 11 macro variables from the proposal, plus RV features from the shared `realized_variance.csv`. Row `t` holds information up to month `t`; `y_target` is `log_rv_1m` of month `t+1`.

In [ ]:
# Explicit list: final_macro_data.csv also has DEXUSEU and USEPUINDXD, which are not in the proposal set
FINAL_MACRO_VARS = [
    "T10Y2Y", "DGS3MO", "BAA10Y", "CPIAUCSL", "T5YIE", "T5YIE_MINUS_T10YIE",
    "VIXCLS", "INDPRO", "UNRATE", "UMCSENT", "DCOILWTICO",
]
RV_FEATURE_COLS = ["log_rv_1m", "log_rv_avg_3m", "log_rv_avg_12m"]

macro = pd.read_csv(MACRO_PATH, index_col="date", parse_dates=True)[FINAL_MACRO_VARS]
rv = pd.read_csv(RV_PATH, parse_dates=["date"])

# NaNs are kept: the 3m/12m averages are missing early in each series, and each model drops its own
aligned_data = {}
for ticker in CYCLICAL_TICKERS:
    rv_t = rv.loc[rv["etf"] == ticker, ["date"] + RV_FEATURE_COLS].set_index("date").sort_index()
    y_target = rv_t["log_rv_1m"].shift(-1).rename("y_target")
    aligned_data[ticker] = macro.join(rv_t, how="inner").join(y_target, how="inner")
    df = aligned_data[ticker]
    print(f"{ticker}: {len(df)} rows, {df.index.min().date()} to {df.index.max().date()}")

## Variants

In [ ]:
NO_VIX_MACRO = [c for c in FINAL_MACRO_VARS if c != "VIXCLS"]

FEATURE_SETS = {
    "en_full_macro": FINAL_MACRO_VARS + ["log_rv_1m"],
    "en_no_vix": NO_VIX_MACRO + ["log_rv_1m"],
    "en_vix_only": ["VIXCLS", "log_rv_1m"],
    "en_no_macro": ["log_rv_1m"],
    "en_full_macro_har": FINAL_MACRO_VARS + RV_FEATURE_COLS,
    "en_no_vix_har": NO_VIX_MACRO + RV_FEATURE_COLS,
    "en_vix_only_har": ["VIXCLS"] + RV_FEATURE_COLS,
    "en_no_macro_har": list(RV_FEATURE_COLS),
}
for name, cols in FEATURE_SETS.items():
    print(f"{name:18s} {len(cols):2d} features")

## Walk-forward engine

At each month the scaler and `ElasticNetCV` (5-fold `TimeSeriesSplit`) are fit on all data up to that month, then the selected `alpha` and `l1_ratio` are used for the forecast.

- `sigma2` is the variance of out-of-fold residuals from the same CV splits.
- `fc_level = exp(fc_log + sigma2 / 2)` is the log-normal level forecast, used for QLIKE.

In [ ]:
def walk_forward(model_df, feature_cols, burn_in=BURN_IN, refit_every=REFIT_EVERY, l1_ratios=L1_RATIOS):
    """Expanding-window forecasts for one model and ticker. model_df has no NaNs in feature_cols or y_target."""
    X = model_df[feature_cols].to_numpy()
    y = model_df["y_target"].to_numpy()
    dates = model_df.index
    rows = []
    pipe, sigma2 = None, np.nan
    alpha_ = l1_ratio_ = np.nan

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        for i in range(burn_in, len(model_df)):
            refit = (i - burn_in) % refit_every == 0
            if refit:
                X_train, y_train = X[:i], y[:i]
                scaler = StandardScaler()
                X_scaled = scaler.fit_transform(X_train)

                cv = ElasticNetCV(l1_ratio=l1_ratios, cv=TimeSeriesSplit(n_splits=5),
                                  max_iter=10000, random_state=0).fit(X_scaled, y_train)
                alpha_, l1_ratio_ = cv.alpha_, cv.l1_ratio_
                model = ElasticNet(alpha=alpha_, l1_ratio=l1_ratio_, max_iter=10000, random_state=0)
                model.fit(X_scaled, y_train)

                # Out-of-fold residuals; the first CV chunk is never tested, so only tested rows count
                oof = np.full(len(y_train), np.nan)
                for tr, te in TimeSeriesSplit(n_splits=5).split(X_scaled):
                    oof[te] = clone(model).fit(X_scaled[tr], y_train[tr]).predict(X_scaled[te])
                tested = ~np.isnan(oof)
                sigma2 = float(np.var(y_train[tested] - oof[tested]))
                pipe = Pipeline([("scaler", scaler), ("model", model)])

            pred_log = pipe.predict(X[i:i + 1])[0]
            rows.append({
                "date": dates[i],
                "target_date": dates[i] + pd.offsets.MonthEnd(1),
                "actual_logRV": y[i],
                "actual_RV": np.exp(y[i]),
                "fc_log": pred_log,
                "sigma2": sigma2,
                "fc_level": np.exp(pred_log + 0.5 * sigma2),
                "alpha": alpha_,
                "l1_ratio": l1_ratio_,
            })
    return pd.DataFrame(rows).set_index("date")

## Run

A full run takes about 30 minutes. With `RECOMPUTE_FORECASTS = False` the saved file is loaded instead.

In [ ]:
TIDY_COLS = ["date", "target_date", "actual_logRV", "actual_RV", "fc_log", "sigma2", "fc_level"]
SAVED_PATH = os.path.join(FORECASTS_DIR, "en_forecasts.csv")


def run_all_models():
    tidy_rows = []
    t0 = time.time()
    for model_name, feature_cols in FEATURE_SETS.items():
        for ticker in CYCLICAL_TICKERS:
            model_df = aligned_data[ticker][feature_cols + ["y_target"]].dropna()
            tidy = walk_forward(model_df, feature_cols).reset_index()[TIDY_COLS]
            tidy.insert(0, "etf", ticker)
            tidy.insert(0, "model", model_name)
            tidy_rows.append(tidy)
        print(f"{model_name:18s} done, {time.time() - t0:.0f}s")
    tidy_df = pd.concat(tidy_rows, ignore_index=True)
    os.makedirs(FORECASTS_DIR, exist_ok=True)
    tidy_df.to_csv(SAVED_PATH, index=False)
    return tidy_df


RECOMPUTE_FORECASTS = False

if RECOMPUTE_FORECASTS or not os.path.exists(SAVED_PATH):
    tidy_df = run_all_models()
else:
    tidy_df = pd.read_csv(SAVED_PATH, parse_dates=["date", "target_date"])
print(f"{len(tidy_df)} rows, models: {tidy_df['model'].nunique()}")

## Accuracy

Pooled and per-ticker MSE on log RV and QLIKE on the level forecast, for all variants. Variants are only comparable within the same spec (single-lag vs HAR have different samples).

In [ ]:
def qlike(actual_rv, fc_level):
    ratio = actual_rv / fc_level
    return ratio - np.log(ratio) - 1


def mse_qlike_table(tidy_df):
    d = tidy_df.assign(sq_err_log=(tidy_df["actual_logRV"] - tidy_df["fc_log"]) ** 2,
                       qlike=qlike(tidy_df["actual_RV"], tidy_df["fc_level"]))
    agg = dict(MSE_logRV=("sq_err_log", "mean"), QLIKE=("qlike", "mean"))
    per_ticker = d.groupby(["model", "etf"]).agg(**agg).reset_index()
    pooled = d.groupby("model").agg(**agg).reset_index().assign(etf="pooled")
    long = pd.concat([per_ticker, pooled], ignore_index=True)

    order = dict(index=list(FEATURE_SETS), columns=CYCLICAL_TICKERS + ["pooled"])
    mse = long.pivot(index="model", columns="etf", values="MSE_logRV").reindex(**order)
    ql = long.pivot(index="model", columns="etf", values="QLIKE").reindex(**order)
    return mse, ql


mse_expanding, qlike_expanding = mse_qlike_table(tidy_df)
print("MSE (log RV)"); print(mse_expanding.round(4)); print()
print("QLIKE"); print(qlike_expanding.round(4))